# Célula 1: Conexão com o Snowflake e Extração
Nesta célula, vamos fazer o Python ir até a sua camada Gold e trazer as duas tabelas para a memória.

In [ ]:
import snowflake.connector
import pandas as pd
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import serialization
import os

# 1. Carregar a chave privada do arquivo rsa_key.p8
with open("rsa_key.p8", "rb") as key:
    p_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend()
    )

pkb = p_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# 2. Configurar a conexão com o Snowflake usando a chave em vez de senha
ctx = snowflake.connector.connect(
    user=os.getenv('SNOWFLAKE_USER'),
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    private_key=pkb,
    warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
    database=os.getenv('SNOWFLAKE_DATABASE'),
    schema=os.getenv('SNOWFLAKE_SCHEMA')
)

cur = ctx.cursor()

print("Conexão estabelecida com sucesso usando chave RSA!")

# 3. Baixar os Datasets
print("Baixando Dataset A (Leakage)...")
cur.execute("SELECT * FROM GOLD_DATASET_A_LEAKAGE")
df_a_leakage = cur.fetch_pandas_all()

print("Baixando Dataset B (Point-in-Time)...")
cur.execute("SELECT * FROM GOLD_DATASET_B_POINT_IN_TIME")
df_b_point_in_time = cur.fetch_pandas_all()

print("Dados carregados com sucesso!")

Conexão estabelecida com sucesso usando chave RSA!
Baixando Dataset A (Leakage)...
Baixando Dataset B (Point-in-Time)...
Dados carregados com sucesso!


# Célula 2: A Função de Validação (O Coração do Experimento)
Para provar o training-serving skew, não podemos usar um train_test_split aleatório comum. Como é um problema temporal, precisamos simular o mundo real: treinar com o passado e prever o futuro. Vamos dividir o dataset cronologicamente (Out-of-Time Validation).

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score
import numpy as np

def treinar_e_avaliar(df, nome_cenario):
    # 1. Ordenar o dataset pela linha do tempo (Crucial para MLOps)
    df = df.sort_values('DATA_DO_PEDIDO').reset_index(drop=True)
    
    features = ['SELLER_ORDER_VOLUME', 'SELLER_DELAY_COUNT', 'SELLER_AVG_REVIEW_SCORE']
    target = 'TARGET_HOUVE_ATRASO'
    
    # Preencher eventuais nulos residuais com 0
    df[features] = df[features].fillna(0)
    
    X = df[features]
    y = df[target]
    
    # 2. Separação Temporal: Treinamos com os 80% mais antigos, testamos nos 20% mais recentes
    split_index = int(len(df) * 0.8)
    
    X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
    y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]
    
    # 3. Treinamento do Modelo
    model = RandomForestClassifier(random_state=42, max_depth=5, n_jobs=-1)
    model.fit(X_train, y_train)
    
    # 4. Predições no conjunto de Teste (O "Futuro")
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # 5. Predições no conjunto de Treino (Apenas para ver o Overfitting/Leakage)
    y_train_prob = model.predict_proba(X_train)[:, 1]
    
    # 6. Calcular métricas
    auc_treino = roc_auc_score(y_train, y_train_prob)
    auc_teste = roc_auc_score(y_test, y_prob)
    precisao_teste = precision_score(y_test, y_pred, zero_division=0)
    
    print(f"=== Resultados: {nome_cenario} ===")
    print(f"ROC-AUC (Treino): {auc_treino:.4f}")
    print(f"ROC-AUC (Teste) : {auc_teste:.4f}  <-- Métrica que importa")
    print(f"Precisão (Teste): {precisao_teste:.4f}")
    
    # Mostrar a importância das features para o modelo
    importancias = zip(features, model.feature_importances_)
    print("\nImportância das Features:")
    for feature, imp in importancias:
        print(f"  - {feature}: {imp:.4f}")
    print("-" * 40)

# Célula 3: Executando o Teste A/B
Agora, na terceira célula, é só chamar a função para os dois datasets e observar o comportamento do modelo:

In [4]:
treinar_e_avaliar(df_a_leakage, "CENÁRIO A (Com Vazamento Temporal)")
print("\n")
treinar_e_avaliar(df_b_point_in_time, "CENÁRIO B (Point-in-Time Correto)")

=== Resultados: CENÁRIO A (Com Vazamento Temporal) ===
ROC-AUC (Treino): 0.6740
ROC-AUC (Teste) : 0.6694  <-- Métrica que importa
Precisão (Teste): 0.7500

Importância das Features:
  - SELLER_ORDER_VOLUME: 0.2754
  - SELLER_DELAY_COUNT: 0.3882
  - SELLER_AVG_REVIEW_SCORE: 0.3364
----------------------------------------


=== Resultados: CENÁRIO B (Point-in-Time Correto) ===
ROC-AUC (Treino): 0.6414
ROC-AUC (Teste) : 0.5210  <-- Métrica que importa
Precisão (Teste): 0.0000

Importância das Features:
  - SELLER_ORDER_VOLUME: 0.2939
  - SELLER_DELAY_COUNT: 0.4296
  - SELLER_AVG_REVIEW_SCORE: 0.2764
----------------------------------------


## O que você vai observar nos resultados
No Cenário A, o modelo terá métricas super infladas (altas) na base de treino, mas quando for testar nos 20% de pedidos mais recentes, o desempenho pode cair ou ser artificialmente enviesado, porque a feature SELLER_DELAY_COUNT já continha atrasos que ocorreram após a compra. A importância das variáveis provavelmente ficará totalmente distorcida.

No Cenário B, o modelo terá métricas mais realistas e consistentes entre treino e teste. A importância das features fará sentido causal. É esse contraste empírico que será o gráfico principal do Capítulo 6 do TCC.